In [23]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import STRtree
from pyprojroot import here

In [24]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [25]:
electric = gpd.read_file(RAW_DATA_DIR/"energy/electric_grid_shp/denue_inegi_22_.shp")
centroids = pd.read_csv(INTERIM_DATA_DIR/"municipality_centroids.csv", encoding="utf-8")

The SCIAN code is a standardized numerical system used to classify businesses and economic units by their primary industrial or commercial activity. SCIAN 221111 is assigned to the economic units for energy generation from fossil fuels. SCIAN 221121 is assigned to energy transmission, SCIAN 221122 to energy distribution, and SCIAN 221123 to energy commercialization.

In [26]:
# We construct separate dataframes for electric generation and distribution economic units.
electric["scian"] = electric["clee"].str[5:11]
elec_genera = electric[(electric["scian"] == "221111") & (electric["codigo_act"] == "221111")].copy()
distribution = ["221122", "221121", "221123"]
elec_distri = electric[(electric["scian"].isin(distribution)) & (electric["codigo_act"].isin(distribution))].copy()

In [27]:
centroids_gdf = gpd.GeoDataFrame(
    centroids,
    geometry=gpd.points_from_xy(centroids['long'], centroids['lat']),
    crs='EPSG:4326'
)

In [28]:
distribution_gdf = gpd.GeoDataFrame(
    elec_distri,
    geometry=gpd.points_from_xy(elec_distri['longitud'], elec_distri['latitud']),
    crs='EPSG:4326'
)

In [29]:
generation_gdf = gpd.GeoDataFrame(
    elec_genera,
    geometry=gpd.points_from_xy(elec_genera['longitud'], elec_genera['latitud']),
    crs='EPSG:4326'
)

In [30]:
CRS_M = 'EPSG:6372'
centroids_m = centroids_gdf.to_crs(CRS_M)
dist_m = distribution_gdf.to_crs(CRS_M)
gen_m  = generation_gdf.to_crs(CRS_M)

Shapely's STRtree allows us to analyze vast amounts of data efficiently. It gradually narrows its search, filtering out obvious mismatches and drastically reducing computing resources. 

In [31]:
dist_tree = STRtree(dist_m.geometry.values)
gen_tree  = STRtree(gen_m.geometry.values)

In [32]:
pts = centroids_m.geometry.values  # array of Shapely Points

In [33]:
centroids_m['dist_min_km'] = [
    p.distance(dist_m.geometry.iloc[dist_tree.nearest(p)]) / 1000 for p in pts
]
centroids_m['gen_min_km'] = [
    p.distance(gen_m.geometry.iloc[gen_tree.nearest(p)]) / 1000 for p in pts
]

In [34]:
# Both dist and gen_min_km is the minimum distance towards a distribution or generation site.
result = centroids_m[['CVEGEO', 'dist_min_km', 'gen_min_km']]

In [35]:
result.to_csv(INTERIM_DATA_DIR/'min_dist_energy.csv', index=False)

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
      CVEGEO  dist_min_km  gen_min_km
1600   21145     3.732014   27.600211
2070   29054     2.673387   12.481203
2435   32007    87.532934   94.000112
205     7017    59.759419   70.744429
1103   20043    10.763625   14.980989
509    13037    34.753770    5.314615
595    14067    12.565708   48.711711
601    14060    15.117186   46.328831
1765   21111    45.769242   69.195073
2075   29050    12.824150   14.113737
